# Lesson 2: Data Cleaning, Relational & JSON Sources

**Track:** AI Engineer — Phase 1: Data Foundations & Classical ML  
**Runtime:** Google Colab Free Tier (CPU)  
**Sources:** `.ref/Data-Science-For-Beginners` (05-relational-databases, 06-non-relational, 08-data-preparation)


In [ ]:
import json
import sqlite3
import numpy as np
import pandas as pd


## 1. Relational Ingestion with SQLite


In [ ]:
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)')
conn.execute('CREATE TABLE orders (order_id INTEGER, customer_id INTEGER, amount TEXT)')
conn.executemany('INSERT INTO customers VALUES (?, ?, ?)', [
    (1, 'Asha', ' Colombo '), (2, 'Ravi', 'kandy'), (3, 'Nila', None)
])
conn.executemany('INSERT INTO orders VALUES (?, ?, ?)', [
    (101, 1, '25.50'), (102, 2, '40.00'), (102, 2, '40.00'), (103, 3, 'invalid'), (104, 1, '18.75')
])

query = '''
SELECT o.order_id, c.name, c.city, o.amount
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.id
'''
raw_df = pd.read_sql_query(query, conn)
print(raw_df)


## 2. Deduplication, String Normalization & Median Imputation


In [ ]:
clean_df = (
    raw_df.drop_duplicates(subset=['order_id'], keep='first')
    .assign(
        city=lambda d: d['city'].fillna('unknown').str.strip().str.title(),
        amount=lambda d: pd.to_numeric(d['amount'], errors='coerce'),
    )
)
clean_df['amount'] = clean_df['amount'].fillna(clean_df['amount'].median())
print(clean_df)


## 3. Flattening Nested JSON Payloads


In [ ]:
payload = json.loads('''[
  {"user": "Asha", "profile": {"tier": "pro", "score": 92}},
  {"user": "Ravi", "profile": {"tier": "free", "score": null}}
]''')
json_df = pd.json_normalize(payload)
json_df['profile.score'] = json_df['profile.score'].fillna(json_df['profile.score'].median())
print(json_df)


## Exercise
Impute missing `amount` values using the per-city median instead of the global median.
